In [ ]:
# ============================================
# 01_EDA.ipynb
# EXPLORATORY DATA ANALYSIS
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import cv2
import random
import json

print("="*60)
print("📊 EXPLORATORY DATA ANALYSIS (EDA)")
print("="*60)

# ============================================
# AUTO DETECT DATASET PATH
# ============================================
def find_dataset():
    """Mencari folder dataset secara otomatis"""
    
    # Cek berbagai kemungkinan lokasi
    possible_paths = [
        Path.cwd().parent / "dataset" / "raw",  # dataset/raw/
        Path.cwd().parent / "dataset",  # dataset/
        Path.cwd() / "dataset" / "raw",  # notebook/dataset/raw/
        Path.cwd() / "dataset",  # notebook/dataset/
        Path.cwd().parent / "notebook" / "dataset" / "raw",  # notebook/dataset/raw/
    ]
    
    for path in possible_paths:
        if path.exists():
            # Cek folder Drowsy dan Non Drowsy
            drowsy = path / "Drowsy"
            non_drowsy = path / "Non Drowsy"
            if drowsy.exists() and non_drowsy.exists():
                print(f"✅ Dataset ditemukan di: {path}")
                return path
    
    # Jika tidak ditemukan, cari manual
    print("🔍 Mencari dataset di seluruh folder...")
    for root, dirs, files in os.walk(Path.cwd().parent):
        if "Drowsy" in dirs and "Non Drowsy" in dirs:
            dataset_path = Path(root)
            print(f"✅ Dataset ditemukan di: {dataset_path}")
            return dataset_path
    
    print("❌ Dataset tidak ditemukan!")
    print("\n📁 Struktur yang dicari:")
    print("   dataset/")
    print("   └── raw/")
    print("       ├── Drowsy/")
    print("       └── Non Drowsy/")
    raise FileNotFoundError("Dataset tidak ditemukan!")

# ============================================
# SETUP PATHS
# ============================================
project_root = Path.cwd().parent
dataset_path = find_dataset()
results_dir = project_root / "results" / "plots"
results_dir.mkdir(parents=True, exist_ok=True)

print(f"📁 Project root: {project_root}")
print(f"📁 Results dir: {results_dir}")

# ============================================
# DETEKSI FOLDER KELAS
# ============================================
print("\n[1] Deteksi Folder Kelas...")

drowsy_folder = None
non_drowsy_folder = None

for folder in dataset_path.iterdir():
    if folder.is_dir():
        name_lower = folder.name.lower()
        if 'drowsy' in name_lower and 'non' not in name_lower:
            drowsy_folder = folder
            print(f"   ✅ Folder Drowsy: {folder.name}")
        elif 'non' in name_lower and 'drowsy' in name_lower:
            non_drowsy_folder = folder
            print(f"   ✅ Folder Non Drowsy: {folder.name}")

if not drowsy_folder or not non_drowsy_folder:
    print("❌ Folder kelas tidak ditemukan!")
    print(f"📁 Isi folder {dataset_path}:")
    for item in dataset_path.iterdir():
        if item.is_dir():
            print(f"   - {item.name}")
    raise FileNotFoundError("Folder Drowsy atau Non Drowsy tidak ditemukan!")

# ============================================
# HITUNG JUMLAH GAMBAR
# ============================================
print("\n[2] Distribusi Kelas...")

def count_images(folder):
    return len(list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg')))

drowsy_count = count_images(drowsy_folder)
non_drowsy_count = count_images(non_drowsy_folder)
total_images = drowsy_count + non_drowsy_count

print(f"   - Drowsy: {drowsy_count} gambar ({drowsy_count/total_images*100:.1f}%)")
print(f"   - Non Drowsy: {non_drowsy_count} gambar ({non_drowsy_count/total_images*100:.1f}%)")
print(f"   - Total: {total_images} gambar")

# ============================================
# PLOT DISTRIBUSI KELAS
# ============================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

class_counts = {'Drowsy': drowsy_count, 'Non Drowsy': non_drowsy_count}
colors = ['#ff6b6b', '#4ecdc4']

ax1.bar(class_counts.keys(), class_counts.values(), color=colors)
ax1.set_title('Distribusi Kelas', fontsize=14, fontweight='bold')
ax1.set_xlabel('Kelas')
ax1.set_ylabel('Jumlah Gambar')
for i, (key, value) in enumerate(class_counts.items()):
    ax1.text(i, value + 50, str(value), ha='center', fontweight='bold')

ax2.pie(class_counts.values(), labels=class_counts.keys(), 
        autopct='%1.1f%%', colors=colors, explode=(0.05, 0.05))
ax2.set_title('Persentase Kelas', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(results_dir / 'class_distribution.png', dpi=150)
plt.show()

# ============================================
# SAMPLE GAMBAR
# ============================================
print("\n[3] Sample Gambar...")

def show_samples(folder, class_name, num=4):
    images = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
    if not images:
        print(f"⚠️ Tidak ada gambar di {class_name}")
        return
    
    samples = random.sample(images, min(num, len(images)))
    
    fig, axes = plt.subplots(1, len(samples), figsize=(12, 3))
    if len(samples) == 1:
        axes = [axes]
    
    for idx, img_path in enumerate(samples):
        try:
            img = Image.open(img_path)
            axes[idx].imshow(img)
            axes[idx].set_title(f'{class_name}\n{img.size}', fontsize=10)
            axes[idx].axis('off')
        except Exception as e:
            print(f"❌ Error loading {img_path.name}: {e}")
    
    plt.tight_layout()
    plt.show()

show_samples(drowsy_folder, 'Drowsy')
show_samples(non_drowsy_folder, 'Non Drowsy')

# ============================================
# ANALISIS UKURAN GAMBAR
# ============================================
print("\n[4] Analisis Ukuran Gambar...")

widths, heights = [], []
for folder in [drowsy_folder, non_drowsy_folder]:
    images = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
    for img_path in random.sample(images, min(100, len(images))):
        try:
            img = Image.open(img_path)
            widths.append(img.size[0])
            heights.append(img.size[1])
        except:
            pass

if widths and heights:
    print(f"   - Rata-rata lebar: {np.mean(widths):.0f} px")
    print(f"   - Rata-rata tinggi: {np.mean(heights):.0f} px")
    print(f"   - Rentang lebar: {min(widths)} - {max(widths)} px")
    print(f"   - Rentang tinggi: {min(heights)} - {max(heights)} px")
else:
    print("⚠️ Tidak ada data ukuran gambar!")

# ============================================
# ANALISIS KECERAHAN
# ============================================
print("\n[5] Analisis Kecerahan...")

brightness_data = []
for class_name, folder in [('Drowsy', drowsy_folder), ('Non Drowsy', non_drowsy_folder)]:
    images = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
    for img_path in random.sample(images, min(100, len(images))):
        try:
            img = cv2.imread(str(img_path))
            if img is not None:
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                brightness_data.append({
                    'class': class_name,
                    'brightness': np.mean(gray)
                })
        except:
            pass

if brightness_data:
    df_brightness = pd.DataFrame(brightness_data)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(x='class', y='brightness', data=df_brightness, palette=colors)
    ax.set_title('Distribusi Kecerahan per Kelas', fontsize=14, fontweight='bold')
    ax.set_xlabel('Kelas')
    ax.set_ylabel('Kecerahan (0-255)')
    plt.tight_layout()
    plt.savefig(results_dir / 'brightness_analysis.png', dpi=150)
    plt.show()

    print("\n📊 Statistik Kecerahan:")
    print(df_brightness.groupby('class')['brightness'].describe())

# ============================================
# SIMPAN INFORMASI DATASET
# ============================================
print("\n[6] Menyimpan informasi dataset...")

dataset_info = {
    "dataset_path": str(dataset_path),
    "total_images": total_images,
    "class_counts": class_counts,
    "image_formats": ["jpg", "png", "jpeg"],
    "drowsy_count": drowsy_count,
    "non_drowsy_count": non_drowsy_count
}

with open(results_dir / "dataset_info.json", 'w') as f:
    json.dump(dataset_info, f, indent=4)

print(f"✅ Informasi dataset disimpan di: {results_dir / 'dataset_info.json'}")

# ============================================
# SELESAI
# ============================================
print("\n" + "="*60)
print("✅ EDA SELESAI!")
print("="*60)

print("\n📊 RINGKASAN DATASET:")
print(f"   - Total gambar: {total_images}")
print(f"   - Drowsy: {drowsy_count} gambar")
print(f"   - Non Drowsy: {non_drowsy_count} gambar")